# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets in this dataset
print("Available record sets (by @id):")
for record_set in metadata.record_sets:
    print(f"- {record_set['@id']}: {record_set['name']}")

# For each record set, list its fields and columns by @id
for record_set in metadata.record_sets:
    print(f"\nFields for record set {record_set['name']} ({record_set['@id']}):")
    for field in record_set.get('fields', []):
        print(f"  - Field name: {field['name']} | @id: {field['@id']}")
        # Print columns for each field if available
        for column in field.get('columns', []):
            print(f"    - Column name: {column['name']} | @id: {column['@id']}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Let's collect all record set @id values
record_set_ids = [r['@id'] for r in metadata.record_sets]

# Load all record sets into pandas DataFrames, indexed by record set @id
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Display columns for the first record set as example
first_record_set_id = record_set_ids[0]
print(f"Columns in record set {first_record_set_id}:")
print(dataframes[first_record_set_id].columns.tolist())
dataframes[first_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# For demonstration, we use the first record set and automatically choose a sample numeric field
df = dataframes[first_record_set_id]

# Try to pick a numeric field (float or int type columns)
numeric_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
if numeric_candidates:
    numeric_field = numeric_candidates[0]
    print(f"Using numeric field for analysis: {numeric_field}")
else:
    raise ValueError("No numeric fields detected for EDA.")

threshold = df[numeric_field].mean() if len(df) > 0 else 0
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold} (mean):")
print(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try to find a categorical field to group by (non-numeric and not index)
group_candidates = [col for col in df.columns if df[col].dtype == object and col != numeric_field]
if group_candidates:
    group_field = group_candidates[0]
    print(f"Grouping by: {group_field}")
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"Grouped data by {group_field}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt

# Simple histogram of the numeric field
plt.figure(figsize=(8, 5))
plt.hist(df[numeric_field].dropna(), bins=30, alpha=0.7, color='skyblue', edgecolor='black')
plt.title(f"Distribution of {numeric_field} in {first_record_set_id}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# If grouped by a categorical field, plot boxplot
if 'group_field' in locals():
    plt.figure(figsize=(10, 5))
    df.boxplot(column=numeric_field, by=group_field, grid=False, rot=60)
    plt.title(f"{numeric_field} by {group_field}")
    plt.suptitle("")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

* In this notebook, we demonstrated how to dynamically explore, extract, and analyze data using the [mlcroissant](https://github.com/mlcommons/croissant) library and the Croissant schema.
* The FAIR^2 dataset reveals rich experimental results from mass spectrometry analysis of human extracellular vesicles, with metadata, columns, and normalization handled in a standard workflow.
* The process involves referencing all data entities by their `@id` values (as per schema best practices), facilitating reproducibility and unambiguous access.
* Further analysis can extend from this template by applying additional data transformations, statistical analysis, and advanced visualizations specific to domain research questions.